### RAG end to end pipeline ###

In [1]:
import tqdm
import os
from pathlib import Path
from langchain_community import document_loaders
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\admin\AppData\Local\Temp\ipykernel_19716\838187882.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community import document_loaders
c:\Users\admin\Documents\Personal_documents\Projects\Research-Paper-Brain-RAG-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Function to process PDF documents and extract text
def process_documents(doc_dir):
    doc_dir = "../data"
    all_documents = []
    pdf_directory = Path(doc_dir)
    pdf_files = list(pdf_directory.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf in pdf_files:
        print(f"Processing {pdf}...")

        try:
            loader = PyPDFLoader(str(pdf))
            documents = loader.load()

            ##adding the metadata to the documents

            for doc in documents:
                doc.metadata["source"] = pdf.name

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf}: {e}")   

    print(f"Total documents processed: {len(all_documents)}")
    return all_documents  


In [3]:
### text splitting the documents into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    all_chunks = []
    for doc in documents:
        chunks = text_splitter.split_documents([doc])
        all_chunks.extend(chunks)

    print(f"Total chunks created: {len(all_chunks)}")
    return all_chunks

all_documents = process_documents("../data")
all_chunks = split_documents(all_documents)

Found 2 PDF files in ..\data
Processing ..\data\attention.pdf...
Processing ..\data\research_paper.pdf...
Total documents processed: 42
Total chunks created: 168


### Embedding and VectorStore DB ###

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [5]:
class EmbeddingManager:
    
    def __init__ (self, model_name: str = "all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):    
        try:

            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")

        except Exception as e:
            
            print(f"Error loading model {self.model_name}: {e}")
            raise e

    def generate_embeddings (self, texts: List[str]) -> np.ndarray:
        try:
            embeddings = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise e

embedding_manager = EmbeddingManager()
embedding_manager
print("Embeddings generated successfully.")

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 930.31it/s]


Model loaded successfully. Embedding dimension: 384
Embeddings generated successfully.


In [6]:
class VectorStore:

    def __init__(self, collection_name: str = "pdf_collection", persist_directory: str = "../data/vectorestore/chroma_db"):

    
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings for RAG"})
            print(f"Vector store initialized at {self.persist_directory} with collection '{self.collection_name}'.")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise e

    def add_embeddings(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")


        ids =[]
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)
            metadatas.append(doc.metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())


        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Added {len(documents)} documents to the vector store.")

        except Exception as e:
            print(f"Error adding embeddings to vector store: {e}")
            raise e    

vector_store = VectorStore()
vector_store

Vector store initialized at ../data/vectorestore/chroma_db with collection 'pdf_collection'.


In [7]:
text =[doc.page_content for doc in all_chunks]

embeddings = embedding_manager.generate_embeddings(text)

vector_store.add_embeddings(all_chunks, embeddings)


Batches: 100%|██████████| 6/6 [00:14<00:00,  2.43s/it]


Added 168 documents to the vector store.


In [8]:
class RagRetriever:

    def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManager):
        
        self.vector_store = vector_store
        self.embedding_manager= embedding_manager

    def retrieve (self, query: str, top_k: int = 5, similarity_threshold: float = 0.0) -> List[Dict[str, Any]]:

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["metadatas", "documents", "embeddings"]
            )

            retrieved_docs = []
            for i in range(len(results["ids"][0])):
                doc_id = results["ids"][0][i]
                metadata = results["metadatas"][0][i]
                document_text = results["documents"][0][i]
                embedding = np.array(results["embeddings"][0][i])

                similarity_score = cosine_similarity([query_embedding], [embedding])[0][0]

                if similarity_score >= similarity_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "metadata": metadata,
                        "document": document_text,
                        "similarity_score": similarity_score
                    })

            return retrieved_docs
        except Exception as e:
            print(f"Error occurred while retrieving documents: {e}")
            return []


In [9]:
rag_retriever=RagRetriever(vector_store=vector_store, embedding_manager=embedding_manager)
rag_retriever.retrieve(query="what is weather?")

Batches: 100%|██████████| 1/1 [00:00<00:00, 27.47it/s]


[{'id': '63654502-104f-4ccc-aad7-985a10a8ac8c',
  'metadata': {'page_label': '21',
   'doi': 'https://doi.org/10.48550/arXiv.2607.05391',
   'author': 'Jacky Kwok; Shulu Li; Pranav Atreya; Yuejiang Liu; Yixing Jiang; Chelsea Finn; Marco Pavone; Ion Stoica; Azalia Mirhoseini',
   'creationdate': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1',
   'total_pages': 31,
   'arxivid': 'https://arxiv.org/abs/2607.05391v1',
   'title': 'LLM-as-a-Verifier: A General-Purpose Verification Framework',
   'license': 'http://creativecommons.org/licenses/by/4.0/',
   'page': 20,
   'trapped': '/False',
   'creator': 'arXiv GenPDF (tex2pdf:a6404ea)',
   'producer': 'pikepdf 8.15.1',
   'source': 'research_paper.pdf'},
  'document': 'URLhttps://arxiv.org/abs/2302.04166.\n[44] Yang Liu, Dan Iter, Yichong Xu, Shuohang Wang, Ruochen Xu, and Chenguang Zhu. G-eval:\nNLG evaluation using GPT-4 with better human alignment. InProceedings of the 

In [17]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

model = ChatGroq(api_key=groq_api_key, model="llama-3.3-70b-versatile", temperature=0.2, max_tokens=1024)

In [24]:
def rag_simple (query, retrievr, model, top_k=5):

    retrieved_docs = retrievr.retrieve(query=query, top_k=top_k)

    if not retrieved_docs:
        return "No relevant documents found."

    context = "\n\n".join([f"Document ID: {doc['id']}\nContent: {doc['document']}" for doc in retrieved_docs])

    prompt = f"Answer the following question based on the provided context:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = model.invoke([prompt.format(query=query, context=context)])
    
    return response

In [26]:
ans = rag_simple(query="who is the CEO of Apple?", retrievr=rag_retriever, model=model, top_k=5)

Batches: 100%|██████████| 1/1 [00:00<00:00, 66.61it/s]


In [27]:
print(ans)

content='The provided context does not mention the CEO of Apple. The context appears to be related to attention mechanisms in neural networks and references various research papers, but it does not contain any information about Apple or its CEO. Therefore, I cannot provide an answer to the question based on the given context.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 1285, 'total_tokens': 1344, 'completion_time': 0.21983516, 'completion_tokens_details': None, 'prompt_time': 0.145745844, 'prompt_tokens_details': None, 'queue_time': 0.051469038, 'total_time': 0.365581004}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f55ce-ffa5-7460-a42a-7db9190ec831-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1285, 'output_tokens': 59, 'total_tokens': 1344}
